# Day 1 - Lesson 2 실습: LangChain 첫 만남

> **이론 문서**: `Day1_Lesson2_Theory.md`  
> **선행 실습**: `Day1_Lesson1_Practice.ipynb`  
> **실습 시간**: 약 60분  
> **사용 모델**: GPT-4o-mini

---

## 🎯 이 노트북에서 만들 것

1교시에 짠 OpenAI SDK 코드를 **LangChain 코드로 다시 작성**하면서 차이를 직접 체감하는 시간입니다.

```
1️⃣ 1교시 코드 복습 → LangChain 코드로 변환 비교
       ↓
2️⃣ Models: ChatOpenAI 사용하기
       ↓
3️⃣ Prompts: ChatPromptTemplate으로 변수화
       ↓
4️⃣ Chains: 파이프 `|`로 연결하기
       ↓
5️⃣ LCEL 보너스: stream / batch 한 줄로 사용
       ↓
6️⃣ Few-shot: 예시로 LLM 학습시키기
       ↓
🎁 도전과제: 페르소나 챗봇 · 리뷰 감정 분류기
```

## 📖 이론 매핑

| 이론 문서 섹션 | 이 노트북에서 다루는 코드 |
|---|---|
| 5부. 1교시 코드를 LangChain으로 변환 | Step 1. SDK ↔ LangChain 비교 |
| 2부 ①. Models | Step 2. ChatOpenAI 직접 사용 |
| 2부 ② + 4부. Prompts | Step 3. ChatPromptTemplate |
| 2부 ③ + 3부. Chains & LCEL | Step 4. `prompt \| llm` 체인 |
| 3부. LCEL 4가지 혜택 | Step 5. stream / batch |
| 4부. Few-shot | Step 6. 예시 기반 분류 |
| (3교시 예고) | 도전과제 1: 페르소나 챗봇 |

---

## 💡 이번 실습의 핵심 한 줄

> ```python
> chain = prompt | llm
> result = chain.invoke({"변수": "값"})
> ```
>
> **이 패턴이 5일 코스 내내 사용됩니다.** 오늘 이 한 줄을 손에 익히는 게 전부예요.


---

# 🛠 Step 0. 환경 준비

1교시와 동일하게 Colab/Jupyter 자동 감지로 설정합니다.  
이번엔 **LangChain 패키지**가 추가됩니다.


In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab 환경입니다.")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter 환경입니다.")


✅ 로컬 Jupyter 환경입니다.


In [2]:
# 2️⃣ LangChain 패키지 설치
# - langchain-core: 인터페이스·추상 클래스 (Runnable, ChatPromptTemplate 등)
# - langchain-openai: OpenAI 전용 통합 (ChatOpenAI)
# - langchain: 위 패키지를 묶은 메타 패키지
!pip install -q langchain langchain-openai langchain-core python-dotenv



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 3️⃣ API 키 로드 (1교시와 동일한 패턴)
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 로드 완료" if os.environ.get("OPENAI_API_KEY") else "❌ 키를 못 찾았어요")


✅ API 키 로드 완료


---

# 1️⃣ Step 1. OpenAI SDK ↔ LangChain 비교

> **이론 매핑**: 5부. "1교시 코드를 LangChain으로 변환"

같은 작업을 두 가지 방식으로 짜보고 차이를 눈으로 봅시다.

### 🅰️ 1교시에 짠 OpenAI SDK 방식 (복습)


In [4]:
# 1교시 방식 — 메시지 딕셔너리를 직접 구성
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 데이터 분석가입니다."},
        {"role": "user", "content": "이상치 탐지 방법을 한 줄로 설명해주세요."},
    ],
)
print(response.choices[0].message.content)  # ← 답변까지 3단계 깊이


이상치 탐지 방법은 데이터에서 일반적인 패턴과 벗어난 값을 식별하기 위해 통계적, 기계 학습 또는 규칙 기반 기법을 사용하는 과정입니다.


### 🅱️ 같은 작업, LangChain 방식


In [5]:
# LangChain 방식 — 부품 3개를 만들어 파이프로 연결
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 부품 1: 모델
llm = ChatOpenAI(model="gpt-4o-mini")

# 부품 2: 프롬프트 (변수 자리 {role}, {question})
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}"),
])

# 부품 3: 체인 = 프롬프트 | 모델
chain = prompt | llm

# 호출 — 변수만 채워서
result = chain.invoke({"role": "데이터 분석가", "question": "이상치 탐지 방법을 한 줄로 설명해주세요."})
print(result.content)  # ← 답변까지 1단계!


이상치 탐지 방법은 데이터를 분석하여 일반적인 패턴에서 크게 벗어난 값을 식별하는 기법입니다.


### 🔍 차이 정리

| 비교 항목 | OpenAI SDK | LangChain |
|---|---|---|
| 답변 꺼내기 | `response.choices[0].message.content` | `result.content` |
| 메시지 형식 | `{"role": ..., "content": ...}` 딕셔너리 | `("system", "...")` 튜플 |
| 프롬프트 | 호출 시마다 작성 | **템플릿으로 분리** |
| 페르소나 변경 | 새 코드 작성 | **변수만 변경** |

✅ **결과 확인**: 두 코드의 답변이 비슷하게 나왔다면 성공입니다. 답변 내용은 같고, **코드 구조만 더 깔끔**해진 거예요.


---

# 2️⃣ Step 2. Models — `ChatOpenAI` 깊이 사용하기

> **이론 매핑**: 2부 ①. Models

`ChatOpenAI`는 매우 유연합니다. 세 가지 입력 방식을 모두 지원해요.


In [6]:
# 방식 A: 그냥 문자열 한 줄만 넘기기 (가장 간단)
result = llm.invoke("Python을 한 줄로 설명해줘.")
print("[방식 A 문자열]\n", result.content)


[방식 A 문자열]
 Python은 간결하고 가독성이 좋으며, 다양한 용도로 사용할 수 있는 고급 프로그래밍 언어입니다.


In [7]:
# 방식 B: LangChain 메시지 객체 사용
from langchain_core.messages import SystemMessage, HumanMessage

result = llm.invoke([
    SystemMessage("당신은 친한 친구처럼 반말로 답변하세요."),
    HumanMessage("Python을 한 줄로 설명해줘."),
])
print("[방식 B 메시지 객체]\n", result.content)


[방식 B 메시지 객체]
 Python은 읽기 쉽고 배우기 쉬운 고수준 프로그래밍 언어야.


In [8]:
# 방식 C: OpenAI 스타일 딕셔너리도 그대로 받음 (호환성)
result = llm.invoke([
    {"role": "system", "content": "당신은 대학 교수입니다. 학술 용어로 답하세요."},
    {"role": "user", "content": "Python을 한 줄로 설명해줘."},
])
print("[방식 C 딕셔너리]\n", result.content)


[방식 C 딕셔너리]
 Python은 명료하고 간결한 문법을 갖춘 고급 프로그래밍 언어로, 다양한 프로그래밍 패러다임을 지원하며 광범위한 응용 분야에 사용된다.


✅ **결과 확인**: 같은 모델인데 system 메시지에 따라 톤이 완전히 다르죠?  
> 💡 실무에서는 **방식 B 메시지 객체**나 **ChatPromptTemplate**(다음 Step)을 가장 많이 씁니다.


---

# 3️⃣ Step 3. Prompts — `ChatPromptTemplate`

> **이론 매핑**: 2부 ② + 4부. Prompts / ChatPromptTemplate 깊이

프롬프트를 **재사용 가능한 템플릿**으로 만들어봅시다.

### LangChain의 메시지 역할 표기

| OpenAI 표기 | LangChain 표기 |
|---|---|
| `"role": "system"` | `("system", "...")` |
| `"role": "user"` | `("human", "...")` |
| `"role": "assistant"` | `("ai", "...")` |


In [ ]:
# 프롬프트 템플릿 정의 — {} 안이 채워질 변수
prompt = ChatPromptTemplate.from_messages([
    ("system", "{role} 한국어로 2문장 이내로 답하세요."),
    ("human", "{question}"),
])
print(prompt)


input_variables=[] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='사용자의 요청을 분석하고 다음 구조로만 응답해 1. 한국어로 2문장 이내로 2. 전문용어를 활용'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='시스템 메시지의 활용방법'), additional_kwargs={})]


### 🤔 실행 전에 1분 멈춤

아래 셀은 `prompt.invoke()`만 호출합니다 — LLM은 아직 안 부릅니다.  
실행하기 *전에* 예측해보세요:

- `formatted` 변수에는 어떤 형태의 값이 들어갈까요?
  - 그냥 채워진 **문자열 하나**?
  - **리스트** 형태?
  - 다른 **객체** 형태?
- `("system", "당신은 {role}입니다...")`에서 `{role}`은 어떻게 변할까요?

예측을 머릿속에 적은 뒤 셀을 실행하고, `type(formatted)`까지 같이 확인하세요.

In [12]:
# 변수만 채우고 LLM은 호출하지 않음 — 디버깅 기법
formatted = prompt.invoke({
    "role": "사용자의 요청을 분석하고 다음 구조로만 응답해 1. 한국어로 2문장 이내로 2. 전문용어를 활용",
    "question": "p-value가 뭐예요?"
})

# 🔍 예측 확인: formatted의 정체는?
print("📦 formatted의 타입:", type(formatted))
print()
print(formatted)

📦 formatted의 타입: <class 'langchain_core.prompt_values.ChatPromptValue'>

messages=[SystemMessage(content='사용자의 요청을 분석하고 다음 구조로만 응답해 1. 한국어로 2문장 이내로 2. 전문용어를 활용', additional_kwargs={}, response_metadata={}), HumanMessage(content='시스템 메시지의 활용방법', additional_kwargs={}, response_metadata={})]


✅ **결과 확인**: `SystemMessage(content="당신은 데이터 분석가...")`처럼 변수가 채워진 메시지 객체가 보이면 성공.  
> 💡 이게 바로 위에서 본 `방식 B 메시지 객체`와 같은 형태예요. 프롬프트 템플릿은 결국 **메시지 객체 리스트를 자동으로 만들어주는 도구**입니다.


---

# 4️⃣ Step 4. Chains — 파이프 `|`로 연결

> **이론 매핑**: 2부 ③ + 3부. Chains & LCEL

이제 두 부품을 `|`로 연결합니다. **이번 강의 전체에서 가장 중요한 한 줄**이에요.

### 직관적 이해: 리눅스 파이프와 같은 개념

```bash
ls -la | grep "py" | head -5
```

> 왼쪽 출력 → 오른쪽 입력. LangChain도 똑같이 동작합니다.


In [ ]:
# ⭐ 핵심 패턴 — 이게 전부입니다
chain = prompt | llm

# 호출 — 프롬프트의 변수만 채우면 끝
result = chain.invoke({
    "role": "친절한 데이터 분석가",
    "question": "표본 평균과 모평균의 차이가 뭐예요?"
})
print(result.content)


### 🔄 같은 체인, 변수만 바꿔서 재사용


In [15]:
# 페르소나 3개 + 본인이 만들 1개 — 같은 체인에 흘려보기
roles = [
    "초등학교 선생님",
    "대학 교수",
    "친한 친구",
    # 🎯 미션: 본인만의 4번째 페르소나를 추가하세요.
    #         (예시: "꼬치꼬치 캐묻는 시니어 리뷰어", "츤데레 데이터 멘토" 등)
    #         조건: 단순 한 단어가 아니라 *성격이 드러나는 묘사*로 작성
    "____",
]

for role in roles:
    result = chain.invoke({"role": role, "question": "표본 평균과 모평균의 차이가 뭐예요?"})
    print(f"\n🎭 [{role}]\n{result.content}\n" + "-"*50)


🎭 [초등학교 선생님]
표본 평균과 모평균은 통계학에서 중요한 개념입니다. 이 두 용어의 차이점을 간단히 설명해드릴게요.

1. **모평균 (Population Mean)**:
   - 모평균은 전체 모집단에서의 평균을 의미합니다. 즉, 연구하고자 하는 전체 집단의 모든 요소를 포함한 수치입니다. 모든 데이터를 수집할 수 있는 경우에는 모평균을 구할 수 있지만, 일반적으로는 불가능한 경우가 많습니다.

2. **표본 평균 (Sample Mean)**:
   - 표본 평균은 모집단의 일부인 표본에서 계산한 평균입니다. 즉, 전체 집단의 일부를 랜덤으로 선택해 그 데이터의 평균을 구한 값입니다. 표본 평균은 모평균을 추정하기 위해 사용됩니다.

**정리하자면**, 모평균은 전체 집단의 평균이고, 표본 평균은 그 집단의 일부(표본)에서 구한 평균입니다. 통계학에서는 표본 평균을 통해 모평균을 추정하고, 이를 기초로 다양한 분석을 진행합니다.
--------------------------------------------------

🎭 [대학 교수]
표본 평균과 모평균은 통계학에서 중요한 개념입니다. 이들의 차이를 다음과 같이 설명할 수 있습니다.

1. **모평균 (Population Mean)**:
   - 모평균은 모집단 전체의 평균값을 말합니다. 모집단은 연구의 대상이 되는 모든 개체나 관찰값의 집합을 의미합니다.
   - 모평균은 일반적으로 µ(뮤)로 표기하며, 모집단의 모든 값의 합을 개체의 수로 나누어서 계산합니다. 즉, \( \mu = \frac{\Sigma X}{N} \) (여기서 \( \Sigma X \)는 모든 관측값의 합, \( N \)은 모집단의 크기입니다.)

2. **표본 평균 (Sample Mean)**:
   - 표본 평균은 모집단의 일부인 표본의 평균값을 의미합니다. 표본은 모집단에서 선택된 개체들의 집합으로, 전체 모집단을 대표하도록 마련된 것입니다.
   - 표본 평균은 일반적으로 x̄(엑스바)로 표기하며, 표본의 모든 값의 합

✅ **결과 확인**: **체인 정의는 1번**, **호출은 4번** — 같은 작업을 반복할 때 LangChain이 얼마나 깔끔한지 보입니다.

> 💡 1교시 같은 페르소나 비교 코드와 비교해보세요. messages 리스트를 매번 새로 만들 필요 없이 변수만 바꾸면 됩니다.

### 🔍 본인 페르소나는 어땠나요?

- 본인이 만든 4번째 페르소나의 답변이 주어진 3개와 *얼마나 다르게* 나왔나요?
- 밍밍하게 나왔다면, *역할 + 말투 + 어휘 스타일* 중 무엇이 부족했을까요?
- 한 단어짜리 페르소나("교수")와 묘사가 풍부한 페르소나("학생 실수에 인내심 없는 늙은 교수") 중 어느 쪽이 더 캐릭터가 살아있나요?

---

### 🐛 잠깐, 이상한 코드 하나 — 변수 이름 디버깅

다음 코드는 **에러가 납니다**. 에러 메시지를 *읽기 전에*, 코드만 보고 어디가 문제일지 30초만 노려보세요.

> 💡 LangChain에서 가장 흔한 실수 1순위입니다. 이 패턴을 한 번 겪어두면 같은 실수 안 합니다.

In [ ]:
# 🐛 이 코드는 에러가 납니다. 어디가 문제일까요?
buggy_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}"),
])

buggy_chain = buggy_prompt | llm

# 호출 — 무슨 에러가 날지 예측해본 뒤 실행하세요
result = buggy_chain.invoke({"persona": "데이터 분석가", "q": "AI가 뭐예요?"})
print(result.content)

🤔 **관찰 포인트**:
- 어떤 종류의 에러가 났나요? (`KeyError`? `ValueError`? 다른 것?)
- 에러 메시지에 등장한 *변수 이름*은 무엇인가요?
- 템플릿이 기대한 변수와 `invoke()`에 넘긴 딕셔너리 키, 어디가 어긋났나요?

> 💡 **교훈**: LangChain 템플릿의 변수 이름은 `invoke()` 딕셔너리 키와 *완벽히 일치*해야 합니다.  
> 1교시 분류기는 "엔진은 도는데 결과가 이상한" 조용한 버그였다면,  
> 이번 건 *아예 안 도는* 시끄러운 버그네요. 에러 메시지를 읽는 훈련이 핵심입니다.

#### ✏️ 직접 고쳐보기

위 셀을 *복사해서* 아래에 새 셀을 만들고 변수 이름을 맞춰서 동작하게 고쳐보세요.

---

# 5️⃣ Step 5. 모델 교체의 위력

> **이론 매핑**: 2부 ①. Models — 모델 교체

LangChain의 진짜 매력은 **모델 교체가 import 한 줄**이라는 점입니다.

### 같은 체인에 다른 모델 끼우기 (OpenAI 내에서)


In [ ]:
# gpt-4o-mini → gpt-4o로 업그레이드 (한 줄 변경)
llm_smart = ChatOpenAI(model="gpt-4o", temperature=0)

chain_smart = prompt | llm_smart  # 체인 코드는 동일!

result = chain_smart.invoke({
    "role": "통계학 교수",
    "question": "베이즈 정리를 한 줄로 설명해주세요."
})
print(result.content)


### 🔄 다른 제공자(Anthropic Claude)로 바꾸려면?

코드 변경은 **딱 두 줄**입니다. (실행은 Anthropic API 키가 있을 때만)

```python
# from langchain_openai import ChatOpenAI       # ← 이걸
# llm = ChatOpenAI(model="gpt-4o-mini")          # ← 이걸

from langchain_anthropic import ChatAnthropic   # ← 이렇게
llm = ChatAnthropic(model="claude-sonnet-4")    # ← 이렇게

# 아래 체인·호출 코드는 한 글자도 안 바뀜
chain = prompt | llm
result = chain.invoke({"role": "...", "question": "..."})
```

> 💡 OpenAI SDK였다면 라이브러리도, 함수도, 응답 객체 구조도 모두 다릅니다.  
> LangChain은 이걸 **공통 인터페이스(Runnable)** 로 통일했어요.


---

# 6️⃣ Step 6. LCEL 보너스 — `stream`과 `batch`

> **이론 매핑**: 3부. "자동으로 따라오는 4가지 혜택"

`|`로 체인을 만들면 다음 4가지를 **공짜로** 얻습니다.

| 메서드 | 용도 |
|---|---|
| `invoke()` | 단일 호출 (이미 사용) |
| `stream()` | 토큰 단위로 실시간 출력 (ChatGPT처럼) |
| `batch()` | 여러 입력을 병렬 처리 |
| `ainvoke()` 등 | 위 메서드의 비동기 버전 |

### 🌊 stream — 답변을 글자 단위로 실시간 출력


In [ ]:
# 토큰이 생성되는 대로 즉시 출력 — ChatGPT 사이트 같은 효과
for chunk in chain.stream({"role": "데이터 분석가", "question": "선형 회귀를 3문장으로 설명해주세요."}):
    print(chunk.content, end="", flush=True)


✅ **결과 확인**: 글자가 한 번에 안 나오고 **타이핑되듯이** 흘러나왔나요?

> 💡 챗봇 UI에서 사용자 체감 속도를 크게 높여주는 기법입니다. Day 1 5교시 Gradio에서 다시 만납니다.

### 📦 batch — 여러 입력을 한 번에


In [ ]:
# 페르소나 3개를 동시에 호출 (서버에서 병렬 처리)
inputs = [
    {"role": "초등 선생님", "question": "확률이 뭐예요?"},
    {"role": "대학 교수", "question": "확률이 뭐예요?"},
    {"role": "친한 친구", "question": "확률이 뭐예요?"},
]

results = chain.batch(inputs)

for inp, res in zip(inputs, results):
    print(f"\n🎭 [{inp['role']}]\n{res.content}\n" + "-"*50)


✅ **결과 확인**: 위에서 `for`로 돌렸을 때보다 **빠르게** 끝났을 거예요. 3번 호출을 병렬로 처리합니다.

> 💡 대량 데이터 처리(고객 문의 1000건 분류 등)에 필수입니다.


---

# 7️⃣ Step 7. Few-shot — 예시로 LLM 학습시키기

> **이론 매핑**: 4부. "Few-shot: 예시로 학습시키기"

분류·추출 작업에서는 **지시만 주는 것보다 예시 2-3개를 보여주는** 게 훨씬 정확합니다.

### Zero-shot vs Few-shot 비교


In [ ]:
# 🅰️ Zero-shot — 지시만 줌
zero_shot = ChatPromptTemplate.from_messages([
    ("system", "리뷰의 감정을 positive / negative / neutral 중 하나로만 답하세요."),
    ("human", "{review}"),
])

# 🅱️ Few-shot — 예시 대화 2개를 끼워 넣음
few_shot = ChatPromptTemplate.from_messages([
    ("system", "리뷰의 감정을 positive / negative / neutral 중 하나로만 답하세요."),
    ("human", "이 카메라 정말 좋아요!"),     # ← 예시 입력
    ("ai", "positive"),                        # ← 예시 답변
    ("human", "포장이 너무 아쉽네요."),
    ("ai", "negative"),
    ("human", "{review}"),                     # ← 실제 입력
])


### 🎲 예측 게임 — 어느 쪽이 어떻게 다를까

같은 리뷰를 zero-shot 체인과 few-shot 체인에 각각 흘려보냅니다.  
실행 *전에*, 어느 쪽이 다음 특성을 가질지 예상해보세요:

| 특성 | Zero-shot | Few-shot |
|---|:---:|:---:|
| 답변 길이가 더 *짧을* 것 | ☐ | ☐ |
| 답변 형식이 더 *일관될* 것 | ☐ | ☐ |
| "neutral입니다" 같은 부연 설명이 붙을 것 | ☐ | ☐ |
| 두 번 돌렸을 때 답이 더 *재현될* 것 | ☐ | ☐ |

체크 표시를 한 뒤 셀을 실행하고, 같은 셀을 **한 번 더** 실행해서 재현성도 확인하세요.

In [ ]:
# 같은 리뷰를 두 체인에 흘려서 답변 형식 비교
review = "배송은 빨랐는데 품질이 기대만큼은 아니었어요."

# 분류 작업은 temperature=0이 정석
llm_strict = ChatOpenAI(model="gpt-4o-mini", temperature=0)

zero_result = (zero_shot | llm_strict).invoke({"review": review})
few_result = (few_shot | llm_strict).invoke({"review": review})

print(f"🅰️ Zero-shot 결과: {zero_result.content}")
print(f"🅱️ Few-shot  결과: {few_result.content}")


✅ **결과 확인**: Zero-shot은 가끔 "neutral입니다" 같은 부연을 붙이는데, Few-shot은 **`negative` 한 단어만** 깔끔하게 나옵니다.

> 💡 핵심: **예시 형식 그대로 답한다**. 분류기를 만들 땐 거의 항상 Few-shot을 씁니다.


---

# 📝 정리: 이번 실습에서 만든 것

- [x] OpenAI SDK ↔ LangChain 코드 변환
- [x] `ChatOpenAI`로 LLM 호출 (3가지 입력 방식)
- [x] `ChatPromptTemplate`으로 프롬프트 변수화
- [x] `prompt | llm` 체인 만들기
- [x] 같은 체인에 모델 교체 (한 줄로)
- [x] `stream` / `batch`로 LCEL 보너스 체험
- [x] Few-shot 패턴으로 분류 정확도 ↑

### 🔑 외워둘 한 줄 패턴

```python
chain = prompt | llm
result = chain.invoke({"변수": "값"})
print(result.content)
```

### ➡️ 다음 시간 예고

다음 3교시는 **자율 실습 시간**입니다.  
아래 도전과제 1번이 그대로 3교시 시작점이에요. 미리 만들어두면 다음 시간이 훨씬 편합니다.

---

이제 도전과제로 손에 익혀봅시다 🎯


---

# 🎁 도전과제

## 도전과제 1. 페르소나 챗봇 v1 (3교시 예고)

다음 3교시에 만들 챗봇의 **첫 버전**입니다.  
페르소나 이름을 받으면 그에 맞는 답변을 돌려주는 함수를 완성하세요.

**힌트**:
- `ChatPromptTemplate.from_messages([...])`로 템플릿 정의
- system 메시지에 `{persona}` 변수 자리 만들기
- `chain = prompt | llm` 한 줄로 체인 구성


In [ ]:
# 📌 페르소나 챗봇 — 빈칸을 채우세요
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

# 👇 빈칸 1: 프롬프트 템플릿에 {persona}, {question} 변수 자리 만들기
persona_prompt = ChatPromptTemplate.from_messages([
            ("system", "당신은 {persona}입니다. 해당 역할에 맞는 말투로 답하세요."),
            ("human", "{question}"),    # ← 빈칸
])

# 👇 빈칸 2: 체인 만들기
persona_chain = persona_prompt | llm


def ask_persona(persona: str, question: str) -> str:
    """페르소나와 질문을 받아 답변 반환"""
    result = persona_chain.invoke({"persona": persona, "question": question})
    return result.content


# 테스트
print(ask_persona("주니어 데이터 분석가에게 따뜻하게 설명하는 멘토", "EDA가 뭐예요?"))
print("\n" + "="*100 + "\n")
print(ask_persona("냉정하고 디테일에 까다로운 시니어 리뷰어", "EDA가 뭐예요?"))


In [ ]:
# 👇 빈칸 1: 프롬프트 템플릿에 {persona}, {question} 변수 자리 만들기
persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {persona}입니다. 이 페르소나에 어울리는 말투와 태도로 답변하세요."),
    ("human", "{question}"),
])

# 👇 빈칸 2: 체인 만들기
persona_chain = persona_prompt | llm

✅ **결과 확인**: 같은 질문이지만 페르소나에 따라 톤이 완전히 달라야 합니다.

> 💡 3교시에는 여기에 **여러 페르소나 사전(dict)** + **사용자 선택 UI**를 붙여서 진짜 챗봇으로 발전시킵니다.

---

## 도전과제 2. 리뷰 감정 분류기 — Few-shot 예시 선택하기

쇼핑몰 리뷰 5개를 batch로 한 번에 분류합니다.  
이번에는 **3가지 few-shot 예시 세트** 중 어느 게 가장 좋은가 판단해보세요.

- 셋 다 돌려서 결과를 비교한 뒤, *근거를 들어* 가장 좋은 세트를 고릅니다.
- 실무에서 분류기 품질 개선은 결국 *어떤 예시를 보여줄지* 고르는 작업입니다.

In [ ]:
# 🎯 같은 시스템 프롬프트, 예시 개수만 바꿔서 비교
# 핵심 트릭: 시스템 프롬프트에는 "짧은 라벨로 답하라"고만 하고
# 라벨 종류 자체를 예시로만 학습시킵니다 → few-shot 효과가 명확히 드러남

candidates_examples = {
    "A (zero-shot, 예시 0개)": [],
    "B (few-shot 2개 — 극단 케이스만)": [
        ("human", "정말 최고! 친구들한테도 다 추천했어요"),
        ("ai", "🌟VIP관리"),
        ("human", "돈 아까워요. 바로 환불 신청했습니다."),
        ("ai", "🚨회수필요"),
    ],
    "C (few-shot 4개 — 중간 케이스 포함)": [
        ("human", "정말 최고! 친구들한테도 다 추천했어요"),
        ("ai", "🌟VIP관리"),
        ("human", "돈 아까워요. 바로 환불 신청했습니다."),
        ("ai", "🚨회수필요"),
        ("human", "디자인은 좋은데 내구성이 좀 아쉬워요"),
        ("ai", "⚠️품질개선"),
        ("human", "그냥 평범해요. 가격은 적당한 듯"),
        ("ai", "😐일반고객"),
    ],
}

llm_classifier = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=15)

reviews = [
    "정말 만족스러운 구매였어요. 또 살래요!",
    "디자인은 예쁜데 한 달 만에 고장났어요.",
    "최악이에요. 환불 신청했습니다.",
    "그냥 평범해요. 가격 대비 적당한 듯.",
    "재구매 의사 100%! 주변에도 다 추천했어요!",
]

inputs = [{"review": r} for r in reviews]

for key, examples in candidates_examples.items():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 쇼핑몰 리뷰 분류 봇입니다. 짧은 라벨 하나로만 답하세요."),
        *examples,
        ("human", "{review}"),
    ])
    chain = prompt | llm_classifier
    results = chain.batch(inputs)

    print(f"\n=== {key} ===")
    for r, review in zip(results, reviews):
        print(f"[{r.content:>12}] {review}")

### 🤔 판단해보기

세 결과를 나란히 놓고 *2-3줄로* 답해보세요:

1. 5개 리뷰 모두에 대해 가장 *깔끔한 한 단어*만 답한 건 어느 세트인가? (A / B / C)
2. "그냥 평범해요" 같은 *애매한 리뷰*에서 세 세트의 답이 어떻게 달랐나요?
3. C에서 `neutral` 예시를 추가했을 때 어떤 점이 좋아졌고, 어떤 부작용이 있었나요?

> 💡 **이 연습의 진짜 의미**: Few-shot은 "예시를 *넣었나* 안 넣었나"가 아니라  
> *어떤 예시를 골랐나*에서 품질이 결정됩니다.  
> 실무에서 분류기 정확도를 끌어올리는 작업은 결국 **좋은 예시를 큐레이션하는 일**입니다.

✅ **결과 확인**: 5개 리뷰가 각각 적절한 감정 라벨로 분류됐나요?

> 💡 이 패턴이 실무에서 가장 많이 쓰이는 LLM 활용법 중 하나입니다.  
> 고객 문의 라우팅, 리뷰 모니터링, 콘텐츠 모더레이션 — 전부 이 구조로 만들 수 있어요.

---

## 🏁 모두 풀었다면

축하합니다! Day 1 Lesson 2 완료 🎉  
다음 3교시 **자율 실습** 노트북에서 페르소나 챗봇을 본격적으로 키워봅니다.
